In [ ]:
from pathlib import Path
from nilearn import plotting, image
from IPython.display import display, Image
import numpy as np
import pandas as pd
from dotenv import dotenv_values

In [ ]:
# --- Set these to match the run you want to QC ---
SUB       = 'sub-03'
RUN_LABEL = 'cap1_run1'   # e.g. 'run1', 'cap1_run1', 'cap2_run2'
 
DIR = Path('1_directory.txt').read_text().strip()
 
# Load pipeline vars for this run
config = dotenv_values(f'{DIR}derivatives/{SUB}/stats/pipeline_vars_{RUN_LABEL}.env')
 
PREPROC_DIR = config.get('PREPROC_DIR') or None
T1_DIR      = config.get('T1_DIR') or None
 
# Fallback if env file predates PREPROC_DIR being added to the wrapper
CAP_NAME = config.get('CAP_NAME') or ''
if not PREPROC_DIR:
    if CAP_NAME:
        PREPROC_DIR = f'{DIR}derivatives/{SUB}/preproc/{CAP_NAME}'
    else:
        PREPROC_DIR = f'{DIR}derivatives/{SUB}/preproc'
    print(f'WARNING: PREPROC_DIR not in env file, derived as: {PREPROC_DIR}')
if not T1_DIR:
    T1_DIR = f'{DIR}derivatives/{SUB}/preproc/T1'
    print(f'WARNING: T1_DIR not in env file, derived as: {T1_DIR}')
mc_file     = config.get('mc_file')
sc_file     = config.get('sc_file')
fiach_file  = config.get('fiach_file')
reg_file    = config.get('reg_file')
smooth_file = config.get('smooth_file')
 
# --- Paths ---
anat     = f'{T1_DIR}/Masked_UNI.nii'
nordic   = f'{PREPROC_DIR}/NORDIC/{mc_file}.nii'
mc       = f'{PREPROC_DIR}/mc/{mc_file}_mcf.nii.gz'
fiach    = f'{PREPROC_DIR}/FIACH/{reg_file}.nii'
reg      = f'{PREPROC_DIR}/reg/reg_{reg_file}.nii.gz'
norm     = f'{PREPROC_DIR}/reg/norm_{reg_file}.nii'
smooth   = f'{PREPROC_DIR}/smooth/{smooth_file}_smooth.nii.gz'
mask_epi = f'{PREPROC_DIR}/FIACH/rfBrainMask.nii'
 
print(f'{SUB} / {RUN_LABEL}')
for label, path in [('anat', anat), ('nordic', nordic), ('mc', mc),
                    ('fiach', fiach), ('reg', reg), ('norm', norm),
                    ('smooth', smooth), ('mask_epi', mask_epi)]:
    exists = '✓' if Path(path).exists() else '✗ MISSING'
    print(f'  [{exists}] {label}: {path}')

### Realignment and unwarping
Realignment plots

NORDIC vs NORDIC_mcf_unwarp

In [ ]:
# Realignment plots
trans_plot = f"{PREPROC_DIR}/mc/{mc_file}_mcf_trans.png"
rot_plot   = f"{PREPROC_DIR}/mc/{mc_file}_mcf_rot.png"
disp_plot  = f"{PREPROC_DIR}/mc/{mc_file}_mcf_disp.png"

for img in [trans_plot, rot_plot, disp_plot]:
    display(Image(img, width=800))

In [ ]:
# NORDIC (before motion correction)
NORDICfunc    = f'{PREPROC_DIR}/NORDIC/{mc_file}.nii'
meanNORfunc   = image.mean_img(NORDICfunc)

# Motion corrected (+ unwarped if fmap exists) — sc_file = mc_file_mcf or mc_file_mcf_unwarp
NORmcufunc     = f'{PREPROC_DIR}/mc/{sc_file}.nii.gz'
meanNORmcufunc = image.mean_img(NORmcufunc)

view1 = plotting.view_img(meanNORfunc,    bg_img=False, black_bg=True, title='NORDIC')
view2 = plotting.view_img(meanNORmcufunc, bg_img=False, black_bg=True, title='mc/unwarp')
view  = plotting.view_img(
    meanNORfunc,
    bg_img=meanNORmcufunc,
    title='NORDIC vs mc/unwarp',
    cmap='hot',
    opacity=0.4,
    symmetric_cmap=False
)
display(view1)
display(view2)
display(view)
del view, view1, view2

##### And overlaid on a T1

In [ ]:
view = plotting.view_img(
    meanNORfunc,
    bg_img=anat,
    title='EPI over T1',
    # cmap='gray',       
    opacity=0.2,
    symmetric_cmap=False
)
view 

In [ ]:
del view
view = plotting.view_img(
    meanNORmcufunc,
    bg_img=anat,
    title='EPI over T1',
    # cmap='gray',    
    opacity=0.2,
    symmetric_cmap=False
)
display(view)

### Masking of the brain
1. For analysis

    UNI_T1 vs Masked_UNI

2. FIACH

    Check that rfBrainMask corresponds to meanFunctional

In [ ]:
del view
UNI = f"{T1_DIR}/UNI_T1.nii"
view = plotting.view_img(
    anat,
    bg_img=UNI,
    title='T1 extraction',
    # cmap='gray',    
    opacity=0.3,
    symmetric_cmap=False
)
display(view)

In [ ]:
meanFunc = f"{PREPROC_DIR}/FIACH/meanFunctional.nii"
rfBrain  = f"{PREPROC_DIR}/FIACH/rfBrainMask.nii"
view = plotting.view_img(
    rfBrain,
    bg_img=meanFunc,
    title='rfBrainMask over meanFunctional',
    cmap='jet',    
    opacity=0.3,
    symmetric_cmap=False
)
display(view)
del view

### FIACH outputs

In [ ]:
imgs = [
    'FIACH_GMM_fit.png',
    'FIACH_TSNR_midslice.png',
    'FIACH_NoisyMask_midslice.png',
    'FIACH_QC_corrections_per_frame.png',
    'FIACH_QC_worstvoxel_1.png',
    'FIACH_QC_worstvoxel_2.png',
    'FIACH_QC_worstvoxel_3.png'
]
for img in imgs:
    display(Image(f"{PREPROC_DIR}/FIACH/{img}", width=500))

Also make sure to double check that all data look reasonable in the command line output:

e.g.

FIACH: Finding principal components of noisy voxels...

FIACH: Noisy voxel PCA complete (k=6)

First 6 PCA components explain 67.27% of variance in noisy voxels.

FIACH: Identifying excessive signal changes

FIACH: Predicted maximum signal change (%): 2.00

FIACH: Large signal changes identified

FIACH: Correcting large signal changes

FIACH: Large signal changes corrected, 5533390 datapoints changed (1.312 %)

Interpolated 5533390 bad data points.

Running QC for LargeTempChanges...

Frames with any correction: 129 / 130 (99.2%)

Median / 95th percentile voxel corrections: 0 / 9 frames

Saved 4D NIfTI with 130 frames → /home/asa25/Desktop/test/derivatives/sub-AY/preproc/FIACH/rclean_scNORDIC_sub-AY_Run_BOLD_mcf_unwarp.nii


#### Confound file QC

Make sure there's 12 columns

In [ ]:
reg_conf = np.loadtxt(f"{PREPROC_DIR}/FIACH/multi_reg_FIACH_6PCs_{SUB}_{RUN_LABEL}.txt")
reg_df = pd.DataFrame(reg_conf)
reg_df

### Co-registration (EPI to T1)

In [ ]:
coregepi     = f"{PREPROC_DIR}/reg/reg_{reg_file}.nii.gz"
meancoregepi = image.mean_img(coregepi)
view = plotting.view_img(
    meancoregepi,
    bg_img=anat,
    title='EPI registered to T1',
    # cmap='jet',    
    opacity=0.3,
    symmetric_cmap=False
)
display(view)
del view

### Normalisation

In [ ]:
mniFunc = f"{dir}/reg/norm_{prereg_file}.nii.gz"
meanmnifunc = image.mean_img(mniFunc)  # nilearn does it for you

view = plotting.view_img(
    meanmnifunc,
    bg_img=anat,
    title='mniFunc over native',
    # cmap='gray',        
    opacity=0.3,
    symmetric_cmap=False
)
view2 = plotting.view_img(
    meanmnifunc,
    title='mniFunc over MNI',
    # cmap='gray',        
    opacity=0.3,
    symmetric_cmap=False
)
display(view)  
display(view2)
del view, view2

### Smoothing

In [ ]:
smoothFunc = f"{dir}/smooth/{presmooth_file}_smooth.nii.gz"
meansmooth = image.mean_img(smoothFunc, copy_header=True)  # nilearn does it for you

# Clip negative values to 0 (for better displaying)
meancoregepi_clipped = image.math_img("np.clip(img, 0, None)", img=meancoregepi)
meansmooth_clipped = image.math_img("np.clip(img, 0, None)", img=meansmooth)

view1 = plotting.view_img(meancoregepi_clipped, bg_img=False, black_bg=True, cmap='Greys_r',
                           symmetric_cmap=False, vmin=0, title='EPI before smoothing')
view2 = plotting.view_img(meansmooth_clipped, bg_img=False, black_bg=True, cmap='Greys_r',
                           symmetric_cmap=False, vmin=0, title='Smoothed EPI')

display(view1)
display(view2)
del view1, view2

### tSNR maps

In [ ]:
bold_tsnr = f"{dir}/tsnr/{presc_file}_tsnr.nii.gz"
# nor_tsnr = f"{dir}/tsnr/{prefiach_file}_tsnr.nii.gz"
normcu_tsnr = f"{dir}/tsnr/{prereg_file}_tsnr.nii.gz"
rclean_tsnr = f"{dir}/tsnr/{prereg_file}_tsnr.nii.gz"
reg_tsnr = f"{dir}/tsnr/{presmooth_file}_tsnr.nii.gz"
smooth_tsnr = f"{dir}/tsnr/{presmooth_file}_smooth_tsnr.nii.gz"

# For displaying purposes, images are set to a low value of 0
bold_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=bold_tsnr)
# nor_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=nor_tsnr)
normcu_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=normcu_tsnr)
reg_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=reg_tsnr)
smooth_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=smooth_tsnr)
rclean_tsnr_clipped = image.math_img("np.clip(img, 0, None)", img=rclean_tsnr)


vmin=0
vmax=250

view1 = plotting.view_img(bold_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                           symmetric_cmap=False, vmin=vmin, vmax=vmax, title='Run_BOLD')
# view2 = plotting.view_img(nor_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                        #    symmetric_cmap=False, vmin=vmin, vmax=vmax, title='NORDIC')
view3 = plotting.view_img(normcu_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                           symmetric_cmap=False, vmin=vmin, vmax=vmax, title='realigned_unwarp')
view4 = plotting.view_img(rclean_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                           symmetric_cmap=False, vmin=vmin, vmax=vmax, title='preregistered (FIACH if run)')
view5 = plotting.view_img(reg_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                           symmetric_cmap=False, vmin=vmin, vmax=vmax, title='registered')
view6 = plotting.view_img(smooth_tsnr_clipped, bg_img=False, black_bg=True, cmap='jet',
                           symmetric_cmap=False, vmin=vmin, vmax=vmax, title='smoothed')

display(view1)
# display(view2)
display(view3)
display(view4)
display(view5)
display(view6)

del view1, view3, view4, view5, view6
# view2,